In [1]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import seaborn as sns

# load data
df = pd.read_csv("../../Datasets/adult.csv")   
df = df.drop_duplicates()
df = df.replace("?", np.nan).dropna()

In [9]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# X: only numeric columns 
X = df[["age", "education.num", "hours.per.week"]]

# most clustering algorithms require scaled values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# create clustering model and train it with our data
kmeans = KMeans(n_clusters=4, random_state=42)

# place the cluster value back to the DataFrame
df["cluster"] = kmeans.fit_predict(X_scaled)
df.head(5)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income,cluster,cluster_average_hours
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K,0,38.4
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K,2,40.9
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K,0,38.4
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K,1,35.5
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K,2,40.9


In [3]:
# let's how the cluster values are spread out
df['cluster'].value_counts()

cluster
1    11199
3     8957
0     7707
2     2276
Name: count, dtype: int64

In [4]:
# calculate the average hours per week for each cluster
cluster_hours_average = df.groupby("cluster")["hours.per.week"].mean()

# place the averages back into the DataFrame
df["cluster_average_hours"] = df["cluster"].map(cluster_hours_average)
df.head(10)

,age,workclass,fnlwgt,education,education.num,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income,cluster,cluster_average_hours
1,82,Private,132870,HS-grad,9,Widowed,Exec-managerial,Not-in-family,White,Female,0,4356,18,United-States,<=50K,0,38.420267
3,54,Private,140359,7th-8th,4,Divorced,Machine-op-inspct,Unmarried,White,Female,0,3900,40,United-States,<=50K,2,40.859842
4,41,Private,264663,Some-college,10,Separated,Prof-specialty,Own-child,White,Female,0,3900,40,United-States,<=50K,0,38.420267
5,34,Private,216864,HS-grad,9,Divorced,Other-service,Unmarried,White,Female,0,3770,45,United-States,<=50K,1,35.531030
6,38,Private,150601,10th,6,Separated,Adm-clerical,Unmarried,White,Male,0,3770,40,United-States,<=50K,2,40.859842
7,74,State-gov,88638,Doctorate,16,Never-married,Prof-specialty,Other-relative,White,Female,0,3683,20,United-States,>50K,0,38.420267
8,68,Federal-gov,422013,HS-grad,9,Divorced,Prof-specialty,Not-in-family,White,Female,0,3683,40,United-States,<=50K,0,38.420267
10,45,Private,172274,Doctorate,16,Divorced,Prof-specialty,Unmarried,Black,Female,0,3004,35,United-States,>50K,3,49.873507
11,38,Self-emp-not-inc,164526,Prof-school,15,Never-married,Prof-specialty,Not-in-family,White,Male,0,2824,45,United-States,>50K,3,49.873507
12,52,Private,129177,Bachelors,13,Widowed,Other-service,Not-in-family,White,Female,0,2824,20,United-States,>50K,0,38.420267


In [5]:
df.groupby("cluster", as_index=False)["cluster_average_hours"].mean()

,cluster,cluster_average_hours
0,0,38.420267
1,1,35.531030
2,2,40.859842
3,3,49.873507


In [ ]:
# pip install statsmodels
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

pd.set_option('display.float_format', lambda x: '%.1f' % x)

# X: the numeric columns, including the new cluster feature
# (leave out the target 'income', and the plain 'cluster' number)
X = df[["age", "education.num", "hours.per.week", "cluster_average_hours"]]

# VIF = Variance Inflation Factor
vif_data = pd.DataFrame()
vif_data["feature"] = X.columns

# calculating VIF for each feature
vif_data["VIF"] = [variance_inflation_factor(X.values, i)
                   for i in range(len(X.columns))]

vif_data.sort_values(by="VIF", ascending=False)

,feature,VIF
3,cluster_average_hours,1.8
1,education.num,1.4
2,hours.per.week,1.4
0,age,1.0


In [10]:
y = (df["income"] == ">50K").astype(int)   
df[["cluster_average_hours"]].corrwith(y)

cluster_average_hours   0.3
dtype: float64

### Conclusion

* I performed KMeans (4 clusters) on age, education.num and hours.per.week, and the cluster average of hours.per.week as a new feature.

* All VIF values are below 2, so the new feature does not introduce multicolinearity but I also came to know that VIF only says the feature is not redundunt, not that it is useful.

* Therefore, I checked what is the correlation. The new feature cluster_average_hours has a correlation of 0.3 with income which is not that good.